# Optimizer Test Notebook

This notebook runs several automated tests against `optimizer.optimizer` (assumes `optimizer.py` is in the same directory).
Each test uses different dates, budgets and options to help you validate behavior and debug the model.

Below each run the notebook prints solver status, termination condition and (when available) the solution details.


## `optimizer` function — parameters and possible values

**Function signature (example)**:
```python
out = optimizer.optimizer(start_date, end_date, budget, origin,
                          csv_path=None, city_type_key=None,
                          verbose=False, solver_name='glpk',
                          require_city_in_results=False, min_days_per_city=1)


## **Parameters**

start_date (datetime.date): Trip start date. Must match months present in your CSV price data.

end_date (datetime.date): Trip end date (inclusive). Must be >= start_date.

budget (float): Maximum total budget for flights across the trip.

origin (str): Starting/return city name. Must exist in the flight price dataset.

csv_path (str, optional): Path to CSV with flights/prices. If None, uses a default file in working directory.

city_type_key (str, optional): If provided, the function filters allowed destination cities using results.json key.

verbose (bool): If True, solver output (tee) is printed.

solver_name (str): Name of the solver to use (e.g., 'glpk', 'cbc', 'gurobi' if available).

require_city_in_results (bool): If True and city_type_key is provided but no results.json exists or key missing, raises an error.

min_days_per_city (int): Minimum days to spend in any visited city (except origin).

## **Returned dict keys (when model solved or feasible):**

status: solver status string

termination: solver termination condition

prepared: internal prepared structures (nodes, arcs, months, costs, etc.)

model: the Pyomo model object (useful for inspection)

objective: integer objective value (number of visited cities)

visited_cities: list of visited city names

flights: list of (origin,dest) arcs taken

total_cost: float, rounded total cost computed from monthly costs via z variables

total_cost_by_month: dict mapping (year,month) to cost

total_cost_by_flight: dict mapping (origin,dest) to cost

In [1]:
# Imports and helper printing
from datetime import date
import importlib
import optimizer  # assumes optimizer.py in same directory
from pprint import pprint


In [2]:
# Define a set of tests to run. Each test is a dict of parameters passed to optimizer.optimizer.
tests = [
    {
        "label": "Test 1 - baseline",
        "start": date(2026, 3, 29),
        "end": date(2026, 4, 5),
        "budget": 430,
        "origin": "Madrid",
        "min_days_per_city": 2,
        "city_type_key": "northern_europe",
        "verbose": False
    },
    {
        "label": "Test 2 - larger budget",
        "start": date(2026, 3, 29),
        "end": date(2026, 4, 9),
        "budget": 750,
        "origin": "Madrid",
        "min_days_per_city": 2,
        "city_type_key": None,
        "verbose": False
    },
    {
        "label": "Test 3 - different origin",
        "start": date(2026, 5, 10),
        "end": date(2026, 5, 17),
        "budget": 300,
        "origin": "Oslo",
        "min_days_per_city": 3,
        "city_type_key": "louvain_community_2",
        "verbose": False
    },
    {
        "label": "Test 4 - tight budget",
        "start": date(2026, 3, 28),
        "end": date(2026, 4, 1),
        "budget": 85,
        "origin": "Milan",
        "min_days_per_city": 1,
        "city_type_key": "budget_hubs",
        "verbose": False
    },
]

results = []

for t in tests:
    print('\n' + '='*80)
    print(t["label"])
    print('='*80)
    try:
        out = optimizer.optimizer(
            t["start"], t["end"], budget=t["budget"], origin=t["origin"],
            min_days_per_city=t["min_days_per_city"], city_type_key=t["city_type_key"],
            verbose=t["verbose"]
        )
        print("status:", out.get('status'), "termination:", out.get('termination'))
        if out.get('termination') in ("optimal", "feasible"):
            print("Objective:", out.get('objective'))
            print("Visited:", out.get('visited_cities'))
            print("Flights:", out.get('flights'))
            print("Total cost:", out.get('total_cost'))
            print("Total cost by Month:")
            pprint(out.get('total_cost_by_month'))
            print("Total cost by Flight:")
            pprint(out.get('total_cost_by_flight'))
        else:
            print("No feasible/optimal solution. See solver output above (if verbose=True).")
        results.append((t["label"], out))
    except Exception as e:
        print("Error running test:", e)
        results.append((t["label"], {"error": str(e)}))



Test 1 - baseline
status: ok termination: optimal
Objective: 5
Visited: ['Copenhagen', 'Helsinki', 'Madrid', 'Oslo', 'Stockholm']
Flights: [('Madrid', 'Copenhagen'), ('Copenhagen', 'Stockholm'), ('Stockholm', 'Oslo'), ('Oslo', 'Helsinki'), ('Helsinki', 'Madrid')]
Total cost: 428.87
Total cost by Month:
{(2026, 3): 92.09, (2026, 4): 336.78}
Total cost by Flight:
{('Copenhagen', 'Stockholm'): 49.42,
 ('Helsinki', 'Madrid'): 146.89,
 ('Madrid', 'Copenhagen'): 92.09,
 ('Oslo', 'Helsinki'): 80.24,
 ('Stockholm', 'Oslo'): 60.24}

Test 2 - larger budget
status: ok termination: optimal
Objective: 7
Visited: ['Bucharest', 'Lisbon', 'London', 'Madrid', 'Oslo', 'Rome', 'Stockholm']
Flights: [('Madrid', 'London'), ('London', 'Lisbon'), ('Lisbon', 'Bucharest'), ('Bucharest', 'Oslo'), ('Oslo', 'Stockholm'), ('Stockholm', 'Rome'), ('Rome', 'Madrid')]
Total cost: 734.98
Total cost by Month:
{(2026, 3): 51.12, (2026, 4): 683.86}
Total cost by Flight:
{('Bucharest', 'Oslo'): 133.41,
 ('Lisbon', 'Buchar